# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv("listingss.csv")
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


Loaded: 69351 listings
                 id      host_id  neighbourhood_group  latitude  longitude  \
count  6.935100e+04      69351.0                  0.0   69351.0    69351.0   
mean   1.373703e+17  124212407.1                  NaN      51.5       -0.1   
std    2.651479e+17  137710444.6                  NaN       0.0        0.1   
min    1.391300e+04       4775.0                  NaN      51.3       -0.5   
25%    1.810090e+07   18707182.0                  NaN      51.5       -0.2   
50%    3.395467e+07   60103501.0                  NaN      51.5       -0.1   
75%    5.265645e+07  196040514.5                  NaN      51.5       -0.1   
max    7.123951e+17  478853993.0                  NaN      51.7        0.3   

         price  minimum_nights  number_of_reviews  reviews_per_month  \
count  69351.0         69351.0            69351.0            52571.0   
mean     177.2             6.0               17.5                0.9   
std      412.8            25.7               40.4         

In [3]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


95th percentile price: £499
                   count   mean    std  min   25%    50%    75%    max
room_type                                                             
Entire home/apt  38583.0  164.0   94.4  8.0  95.0  140.0  204.0  499.0
Hotel room         221.0  185.4  106.7  0.0  99.0  198.0  255.0  475.0
Private room     26722.0   63.5   50.2  0.0  35.0   50.0   74.0  499.0
Shared room        398.0   57.5   56.3  7.0  29.0   38.0   63.0  450.0


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

> 💡 `df_cap[df_cap['room_type'].isin(['Entire home/apt','Private room'])]`


In [4]:
# Task 1
# YOUR CODE HERE
# Task 1 — Histogram: price by room type (overlapping distributions)
import pandas as pd
import plotly.express as px

# Load the dataset
df = pd.read_csv("listingss.csv")

# Filter for relevant room types
df_cap = df[df["room_type"].isin(["Entire home/apt", "Private room"])]

# Create the histogram
fig = px.histogram(
    df_cap,
    x="price",
    color="room_type",
    barmode="overlay",
    opacity=0.6,
    nbins=50,
    title="Price Distribution: Entire home/apt vs Private room"
)

# Add vertical lines for medians
for room in ["Entire home/apt", "Private room"]:
    median_price = df_cap[df_cap["room_type"] == room]["price"].median()
    fig.add_vline(
        x=median_price,
        line_dash="dash",
        line_color="red" if room == "Entire home/apt" else "blue",
        annotation_text=f"{room} median: £{median_price:.2f}",
        annotation_position="top"
    )

# Show the figure
fig.show()


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [5]:
# Task 2
# YOUR CODE HERE
# Task 2 — Box plot: listing activity by borough
import pandas as pd
import plotly.express as px

# Load the dataset
df = pd.read_csv("listingss.csv")

# Filter out listings with zero reviews per month
df_active = df[df["reviews_per_month"] > 0]

# Compute median reviews per month by borough
borough_medians = (
    df_active.groupby("neighbourhood")["reviews_per_month"]
    .median()
    .sort_values(ascending=False)
)

# Identify the two most active boroughs
top_two = borough_medians.head(2).index.tolist()

# Add a column to highlight top boroughs
df_active["highlight"] = df_active["neighbourhood"].apply(
    lambda x: "Top 2 Active" if x in top_two else "Others"
)

# Create horizontal box plot
fig = px.box(
    df_active,
    y="neighbourhood",
    x="reviews_per_month",
    color="highlight",
    points="outliers",
    category_orders={"neighbourhood": borough_medians.index.tolist()},
    title=f"Listing Activity by Borough — Top: {top_two[0]} & {top_two[1]}"
)

# Show the figure
fig.show()


C:\Users\Manish\AppData\Local\Temp\ipykernel_12556\3400891097.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_active["highlight"] = df_active["neighbourhood"].apply(
